# NeRF & Gaussian Splatting Reconstruction

**Author:** Yuxi Yang
**Repo:** [github.com/YuxiYang01/nerf-gaussian-reconstruction](https://github.com/YuxiYang01/nerf-gaussian-reconstruction)

## Pipeline
1. Data — Pre-processed COLMAP output in Google Drive (skipped here)
2. Environment — PyTorch + nerfstudio + tiny-cuda-nn + patches
3. Train — Nerfacto and/or Splatfacto
4. Render — Spiral camera path video
5. Evaluate — PSNR / SSIM / LPIPS
6. Export — Point cloud

## Drive Layout
```
MyDrive/
  colmap_workspace/
    nerfstudio_data/         <- DATA_ROOT
      transforms.json        (OPENCV, 23 frames)
      sparse_pc.ply
      images/                (5712x4284)
      images_2/ images_4/ images_8/
  nerf-gaussian-reconstruction/
    runs/                    (checkpoints)
    results/                 (renders, evals, exports)
```

## 0 - Mount Drive & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib, glob, json, site

DATA_ROOT  = "/content/drive/MyDrive/colmap_workspace/nerfstudio_data"
OUTPUT_DIR = "/content/drive/MyDrive/nerf-gaussian-reconstruction/runs"
RESULT_DIR = "/content/drive/MyDrive/nerf-gaussian-reconstruction/results"

for d in [OUTPUT_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

assert pathlib.Path(DATA_ROOT, "transforms.json").exists(), "transforms.json not found!"
print("Data root :", DATA_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Result dir:", RESULT_DIR)

In [ ]:
# Verify data
tf = json.load(open(f"{DATA_ROOT}/transforms.json"))
print(f"Camera model : {tf.get('camera_model')}")
print(f"Resolution   : {tf['w']} x {tf['h']}")
print(f"Num frames   : {len(tf['frames'])}")
print(f"First frame  : {tf['frames'][0]['file_path']}")
print(f"PLY file     : {tf.get('ply_file_path', 'N/A')}")

## 1 - Environment Setup

Patches needed for Colab Python 3.12 + PyTorch 2.6:
- torch.compile crashes on 3.12+ -> no-op patch
- torch.load defaults weights_only=True -> patch for old checkpoints
- tiny-cuda-nn required or renders are blank

In [ ]:
# 1a) PyTorch + nerfstudio
!pip install -q torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 \
    --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q nerfstudio

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
!which ns-train
!which ns-render

In [ ]:
# 1b) tiny-cuda-nn (~5-10 min compile)
!pip install -q ninja
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

import tinycudann
print("tiny-cuda-nn OK")

In [ ]:
# 1c) Patch torch.compile for Python 3.12+
misc_path = f"{site.getsitepackages()[0]}/nerfstudio/utils/misc.py"
with open(misc_path, "r") as f:
    src = f.read()

if 'return torch.compile(*args, **kwargs)' in src and 'sys.version_info' not in src:
    patch = (
        'import sys\n'
        '    if sys.version_info >= (3, 12):\n'
        '        if len(args) == 1 and callable(args[0]):\n'
        '            return args[0]\n'
        '        return lambda fn: fn\n'
        '    return torch.compile(*args, **kwargs)'
    )
    src = src.replace('return torch.compile(*args, **kwargs)', patch)
    with open(misc_path, "w") as f:
        f.write(src)
    print("Patched misc.py")
else:
    print("Already patched")

In [ ]:
# 1d) Patch torch.load for old checkpoints
eval_path = f"{site.getsitepackages()[0]}/nerfstudio/utils/eval_utils.py"
with open(eval_path, "r") as f:
    src = f.read()

old = 'torch.load(load_path, map_location="cpu")'
new = 'torch.load(load_path, map_location="cpu", weights_only=False)'

if old in src and new not in src:
    src = src.replace(old, new)
    with open(eval_path, "w") as f:
        f.write(src)
    print("Patched eval_utils.py")
else:
    print("Already patched")

## 2 - Train Nerfacto

30,000 iterations, ~3-4 hours on T4.
**Skip training cell if you already have a checkpoint.**

In [ ]:
# Check existing checkpoints
existing = sorted(glob.glob(f"{OUTPUT_DIR}/nerfstudio_data/nerfacto/*/config.yml"))
if existing:
    print("Existing nerfacto runs:")
    for r in existing:
        ckpts = glob.glob(os.path.dirname(r) + "/nerfstudio_models/*.ckpt")
        steps = [os.path.basename(c) for c in ckpts]
        print(f"  {r}")
        print(f"    checkpoints: {steps}")
else:
    print("No existing runs. Train below.")

In [ ]:
# TRAIN nerfacto — skip if checkpoint exists
!ns-train nerfacto \
    --data "$DATA_ROOT" \
    --output-dir "$OUTPUT_DIR" \
    --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True

In [ ]:
# Set config path
configs = sorted(glob.glob(f"{OUTPUT_DIR}/nerfstudio_data/nerfacto/*/config.yml"))
if configs:
    NERFACTO_CONFIG = configs[-1]
else:
    NERFACTO_CONFIG = f"{OUTPUT_DIR}/nerfstudio_data/nerfacto/2026-02-23_234919/config.yml"

assert pathlib.Path(NERFACTO_CONFIG).exists(), f"Not found: {NERFACTO_CONFIG}"
print(f"Nerfacto config: {NERFACTO_CONFIG}")

## 3 - Train Splatfacto (Optional)

In [ ]:
# TRAIN splatfacto — skip if not needed
!ns-train splatfacto \
    --data "$DATA_ROOT" \
    --output-dir "$OUTPUT_DIR" \
    --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True

In [ ]:
configs_s = sorted(glob.glob(f"{OUTPUT_DIR}/nerfstudio_data/splatfacto/*/config.yml"))
SPLATFACTO_CONFIG = configs_s[-1] if configs_s else None
print(f"Splatfacto config: {SPLATFACTO_CONFIG}")

## 4 - Render Spiral Video

Renders to local /content/ (Drive writes can produce empty files), then copies.

In [ ]:
# Render nerfacto
!ns-render spiral \
    --load-config "$NERFACTO_CONFIG" \
    --output-path /content/nerfacto_spiral.mp4 \
    --seconds 6 \
    --downscale-factor 4 \
    --eval-num-rays-per-chunk 512

!ls -lh /content/nerfacto_spiral.mp4

In [ ]:
# Copy to Drive
!cp /content/nerfacto_spiral.mp4 "$RESULT_DIR/nerfacto_spiral.mp4"
print("Saved to Drive")

In [ ]:
# Play
from IPython.display import Video, display
display(Video("/content/nerfacto_spiral.mp4", embed=True, width=640))

In [ ]:
# (Optional) Render splatfacto
if SPLATFACTO_CONFIG:
    !ns-render spiral \
        --load-config "$SPLATFACTO_CONFIG" \
        --output-path /content/splatfacto_spiral.mp4 \
        --seconds 6 \
        --downscale-factor 4 \
        --eval-num-rays-per-chunk 512
    !cp /content/splatfacto_spiral.mp4 "$RESULT_DIR/splatfacto_spiral.mp4"
    display(Video("/content/splatfacto_spiral.mp4", embed=True, width=640))
else:
    print("Splatfacto not trained.")

## 5 - Evaluation

In [ ]:
# Eval nerfacto
!ns-eval \
    --load-config "$NERFACTO_CONFIG" \
    --output-path /content/eval_nerfacto.json

eval_file = "/content/eval_nerfacto.json"
if pathlib.Path(eval_file).exists():
    ev = json.load(open(eval_file))
    print("Nerfacto Results:")
    for k, v in ev.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    import shutil
    shutil.copy(eval_file, RESULT_DIR)

In [ ]:
# Eval splatfacto
if SPLATFACTO_CONFIG:
    !ns-eval \
        --load-config "$SPLATFACTO_CONFIG" \
        --output-path /content/eval_splatfacto.json
    eval_file_s = "/content/eval_splatfacto.json"
    if pathlib.Path(eval_file_s).exists():
        ev_s = json.load(open(eval_file_s))
        print("Splatfacto Results:")
        for k, v in ev_s.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        import shutil
        shutil.copy(eval_file_s, RESULT_DIR)
else:
    print("Splatfacto not trained.")

## 6 - Export Point Cloud

In [ ]:
EXPORT_DIR = f"{RESULT_DIR}/exports/nerfacto_pointcloud"
os.makedirs(EXPORT_DIR, exist_ok=True)

!ns-export pointcloud \
    --load-config "$NERFACTO_CONFIG" \
    --output-dir "$EXPORT_DIR" \
    --num-points 100000

!ls -lh "$EXPORT_DIR/"

## Done

In [ ]:
print("=" * 50)
print("  Pipeline Complete")
print("=" * 50)
print(f"  Data       : {DATA_ROOT}")
print(f"  Nerfacto   : {NERFACTO_CONFIG}")
if SPLATFACTO_CONFIG:
    print(f"  Splatfacto : {SPLATFACTO_CONFIG}")
print(f"  Results    : {RESULT_DIR}/")
print("=" * 50)